# Word2Vec (Skip-gram) from Scratch in NumPy

In this notebook, I implement a tiny version of **Word2Vec (Skip-gram)** from scratch using only NumPy.

Goal:
- Convert words into vectors (embeddings)
- Train these vectors so that **words that appear in similar contexts have similar embeddings**
- Use a tiny toy corpus:
  - "I like apples"
  - "I like bananas"
  - "I eat apples"

I focus on:
- A **single hidden layer** neural network
- **Skip-gram** objective: given a **center word**, predict its **context words**
- Training with **softmax + cross-entropy** and **gradient descent**


## 1. Toy Corpus

We use three simple sentences so that we can inspect the math by hand:

- "I like apples"
- "I like bananas"
- "I eat apples"


In [50]:
import numpy as np


In [ ]:

doc1 = "I like apples"
doc2 = "I like bananas"
doc3 = "I eat apples"

documents = [doc1, doc2, doc3]


## 2. Vocabulary and One-Hot Encoding

Next, I build a **vocabulary** of unique words and map each word to an index.

Then I represent each word as a **one-hot vector**:
- Length = `vocab_size`
- Exactly one position is 1 (the word's index), rest are 0

This is what goes into the **input layer**.


In [41]:
vocab = []
for doc in documents:
    for word in doc.split():
        if word not in vocab:
            vocab.append(word)

vocab_size = len(vocab)
word_index = {word: i for i, word in enumerate(vocab)}

vocab, vocab_size, word_index


(['I', 'like', 'apples', 'bananas', 'eat'],
 5,
 {'I': 0, 'like': 1, 'apples': 2, 'bananas': 3, 'eat': 4})

In [42]:
def one_hot_word(word):
    vec = np.zeros(vocab_size, dtype=int)
    vec[word_index[word]] = 1
    return vec

encoded_docs = []
for doc in documents:
    words = doc.split()
    encoded_sentence = [one_hot_word(w) for w in words]
    encoded_docs.append(encoded_sentence)

for doc, enc in zip(documents, encoded_docs):
    print("\nSentence:", doc)
    for w, vec in zip(doc.split(), enc):
        print(w, "->", vec)



Sentence: I like apples
I -> [1 0 0 0 0]
like -> [0 1 0 0 0]
apples -> [0 0 1 0 0]

Sentence: I like bananas
I -> [1 0 0 0 0]
like -> [0 1 0 0 0]
bananas -> [0 0 0 1 0]

Sentence: I eat apples
I -> [1 0 0 0 0]
eat -> [0 0 0 0 1]
apples -> [0 0 1 0 0]


## 3. Skip-gram Training Pairs

I use the **Skip-gram** idea:

> Given a **center word**, predict its **context words** within a window.

With `window_size = 1`, for `"I like apples"` I get pairs like:
- (I → like)
- (like → I)
- (like → apples)
- (apples → like)


In [ ]:
window_size = 1

pairs = []  

for doc in documents:
    words = doc.split()
    for i, center in enumerate(words):
        for j in range(max(0, i-window_size), min(len(words), i+window_size+1)):
            if j != i:
                context = words[j]
                pairs.append((center, context))

print("Total pairs:", len(pairs))
print("Sample pairs:", pairs[:10])


Total pairs: 12
Sample pairs: [('I', 'like'), ('like', 'I'), ('like', 'apples'), ('apples', 'like'), ('I', 'like'), ('like', 'I'), ('like', 'bananas'), ('bananas', 'like'), ('I', 'eat'), ('eat', 'I')]


## 4. Model Architecture

I use a very small neural network:

- Input layer: one-hot vector of size `vocab_size` (V)
- Hidden layer: embedding vector of size `embedding_dim` (D)
- Output layer: scores for each word in the vocabulary (size V)

Two weight matrices:

- `W1` (V × D): input → hidden  
  - each **row** is the embedding of a word  
- `W2` (D × V): hidden → output  
  - each **column** is the output weight vector for a word

There is **no activation** between input and hidden.  
The hidden layer is simply the **embedding lookup**.


In [ ]:
embedding_dim = 2
V = vocab_size
D = embedding_dim

np.random.seed(0)
W1 = np.random.randn(V, D) * 0.01   
W2 = np.random.randn(D, V) * 0.01   

def softmax(u):
    u = u - np.max(u)      
    exp_u = np.exp(u)
    return exp_u / np.sum(exp_u)

def forward(center_vec, W1, W2):
    
    h = center_vec @ W1        
    u = h @ W2                 
    y_hat = softmax(u)         
    return h, u, y_hat

def train_step(center_word, context_word, W1, W2, lr=0.1):
    x = one_hot_word(center_word)      
    y = one_hot_word(context_word)     

    
    h, u, y_hat = forward(x, W1, W2)

    
    loss = -np.log(y_hat[word_index[context_word]] + 1e-12)

    
    du = y_hat - y                     

    
    dW2 = np.outer(h, du)              
    dh = W2 @ du                       
    dW1 = np.outer(x, dh)              

    
    W1 -= lr * dW1
    W2 -= lr * dW2

    return loss, W1, W2


## 5. Training the Model

For each epoch:
- Loop over all (center, context) pairs
- Do a forward pass: input → hidden → output
- Compute loss: **how wrong is the prediction?**
- Backpropagate: update `W1` and `W2` with gradient descent

I train for 200 epochs and track the total loss.


In [45]:
epochs = 200
lr = 0.1

for epoch in range(1, epochs+1):
    total_loss = 0
    for center, context in pairs:
        loss, W1, W2 = train_step(center, context, W1, W2, lr=lr)
        total_loss += loss

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}")


Epoch 50, Loss: 10.5371
Epoch 100, Loss: 10.0334
Epoch 150, Loss: 9.8754
Epoch 200, Loss: 9.7966


## 6. Learned Embeddings and Similarity

After training, each row of `W1` is the **embedding vector** of a word.

I print:
- the raw embedding vectors
- the **most similar words** using cosine similarity

Even with this tiny corpus, we expect:
- "apples" and "bananas" to be somewhat similar
- "I" to relate to "like" and "eat"


In [46]:
print("\nEmbeddings (W1 rows):")
for word in vocab:
    print(word, "->", W1[word_index[word]])

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a)*np.linalg.norm(b) + 1e-12)

def most_similar(word, topn=3):
    v = W1[word_index[word]]
    sims = []
    for w in vocab:
        if w != word:
            sims.append((w, cosine(v, W1[word_index[w]])))
    sims.sort(key=lambda x: x[1], reverse=True)
    return sims[:topn]

for w in vocab:
    print(w, "->", most_similar(w))



Embeddings (W1 rows):
I -> [ 0.96722052 -0.35223605]
like -> [-1.05835014 -0.19697403]
apples -> [ 0.95601458 -1.08665876]
bananas -> [2.89756029 0.22894698]
eat -> [-1.12031415  1.41826178]
I -> [('bananas', np.float64(0.9097581610602523)), ('apples', np.float64(0.8775713794941268)), ('eat', np.float64(-0.850959785820806))]
like -> [('eat', np.float64(0.4658152904788169)), ('apples', np.float64(-0.512006254221305)), ('I', np.float64(-0.8611573908983984))]
apples -> [('I', np.float64(0.8775713794941268)), ('bananas', np.float64(0.599341162171323)), ('like', np.float64(-0.512006254221305))]
bananas -> [('I', np.float64(0.9097581610602523)), ('apples', np.float64(0.599341162171323)), ('eat', np.float64(-0.556123991555758))]
eat -> [('like', np.float64(0.4658152904788169)), ('bananas', np.float64(-0.556123991555758)), ('I', np.float64(-0.850959785820806))]


## 7. Experiment: Training Only on "I like apples"

To better understand the behavior, I restrict training to just one sentence:

- "I like apples"

With `window_size = 1` the Skip-gram pairs are:
- (I → like)
- (like → I)
- (like → apples)
- (apples → like)

I retrain on only these pairs to see how embeddings move.


In [47]:
doc1 = "I like apples"
words = doc1.split()
window_size = 1

pairs_doc1 = []
for i, center in enumerate(words):
    for j in range(max(0, i-window_size), min(len(words), i+window_size+1)):
        if j != i:
            pairs_doc1.append((center, words[j]))

pairs_doc1


[('I', 'like'), ('like', 'I'), ('like', 'apples'), ('apples', 'like')]

## 8. Manual Walkthrough: One Training Step ( "I" → "like" )

In this section, I manually walk through **one** training example:

- Center word: `"I"`
- Target/context word: `"like"`

Steps:
1. Build one-hot vectors for `"I"` (input) and `"like"` (target)
2. Compute hidden vector `h = x · W1`
3. Compute output scores `u = h · W2`
4. Apply softmax to get probabilities
5. Compute loss `-log P("like")`
6. Compute gradients `du`, `dW2`, `dW1`
7. Update `W1` and `W2`
8. Compare the embedding of `"I"` **before vs after** the update

This shows exactly how Word2Vec **moves the embedding of a word** to make it better at predicting its context.


In [48]:
input_target = pairs_doc1[0]
i_vector = encoded_docs[0][0]
like_vector = encoded_docs[0][1]
...
H = i_vector @ W1
U = H @ W2
P = softmax(U)
...
du = P - y
...
print("Before update, embedding('I'):", W1[word_index["I"]])
...
print("After update,  embedding('I'):", W1[word_index["I"]])


Before update, embedding('I'): [ 0.96722052 -0.35223605]
After update,  embedding('I'): [ 0.96722052 -0.35223605]


### How many layers are trained?

This Word2Vec model has:

- **Input layer** (one-hot) – no weights, just a vector
- **Hidden layer** – weights in `W1` (V × D)
- **Output layer** – weights in `W2` (D × V)

We train:
- `W1` → the **word embeddings** (this is what we finally care about)
- `W2` → helper weights used during training

There is effectively **one hidden layer**, so this is a shallow neural network.


### How training works (in simple terms)

For each pair (center_word → context_word):

1. Convert the center word to a one-hot vector.
2. Look up its embedding using `W1`.
3. Use that embedding to predict **all vocabulary words** via `W2` and softmax.
4. Compare the predicted probability of the true context word with 1 (we want it to be high).
5. Compute the error.
6. Slightly update `W1` (only the center word's row) and `W2` using gradient descent.

Repeat this process for **all pairs**, for many epochs.

Over time:
- words that appear in similar contexts move closer in embedding space.
